# 00. MCP 架构模型

MCP (Model Context Protocol) 是 Anthropic 提出的 Agent-Tool 通信标准。核心架构：

- **Client**：宿主应用（Claude Desktop、IDE、你的 Agent）
- **Server**：提供工具、资源、提示词模板的进程
- **Transport**：stdio（子进程 stdin/stdout）或 SSE（HTTP 长连接）
- **Protocol**：JSON-RPC 2.0

> **检查点**：能画出 MCP 的 Client-Server 拓扑图，标注 Transport 和消息流向。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-flash  base_url=https://api.deepseek.com


In [2]:
# MCP 生命周期模拟
import json, subprocess, sys

# 启动一个 toy server 进程
proc = subprocess.Popen(
    [sys.executable, '-c', '''
import json, sys
for line in sys.stdin:
    req = json.loads(line.strip())
    method = req.get("method","")
    mid = req.get("id")
    if method == "initialize":
        print(json.dumps({"jsonrpc":"2.0","id":mid,"result":{"serverInfo":{"name":"toy","version":"1.0"}}}))
    elif method == "tools/list":
        print(json.dumps({"jsonrpc":"2.0","id":mid,"result":{"tools":[{"name":"echo","description":"Echo input"}]}}))
    elif method == "tools/call":
        args = req["params"]["arguments"]
        print(json.dumps({"jsonrpc":"2.0","id":mid,"result":{"content":[{"type":"text","text":f"Echo: {args}"}]}}))
    sys.stdout.flush()
'''],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True
)

def rpc(method, params=None):
    msg = {"jsonrpc":"2.0","id":1,"method":method}
    if params: msg["params"] = params
    proc.stdin.write(json.dumps(msg)+"\n"); proc.stdin.flush()
    return json.loads(proc.stdout.readline())

print("1. Initialize:", rpc("initialize")["result"]["serverInfo"]["name"])
print("2. Tools:", [t["name"] for t in rpc("tools/list")["result"]["tools"]])
resp = rpc("tools/call", {"name":"echo","arguments":{"msg":"hello"}})
print("3. Echo result:", resp["result"]["content"][0]["text"])
proc.terminate()

1. Initialize: toy
2. Tools: ['echo']
3. Echo result: Echo: {'msg': 'hello'}


## MCP 的三个核心原语

| 原语 | 方法 | 说明 |
|------|------|------|
| **Tool** | `tools/list`, `tools/call` | Agent 可调用的函数，带 inputSchema |
| **Resource** | `resources/list`, `resources/read` | 暴露给 Agent 的数据（文件、数据库记录） |
| **Prompt** | `prompts/list`, `prompts/get` | 参数化的提示词模板 |

> **检查点**：能区分 Tool 和 Resource 的使用场景。Tool 是「做什么」，Resource 是「有什么」。

## 真实 API 实验

LLM 扮演 MCP client 的「大脑」：给它 tools/list 的真实返回，让它决定调哪个工具。


In [3]:
# 真实 MCP server + LLM 工具选择
SCRIPT = ROOT / 'agents' / 'protocols' / 'scripts' / 'mcp_client_demo.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--use-api', '--query', '帮我查一下北京的天气'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


Starting MCP server...
Initialize: server=toy-mcp-server v1.0.0
Tools available: ['calculator', 'weather']
LLM selected: weather({"city": "北京"})
Result: Unknown city



## 练习

1. 修改上面的 toy server：增加 `sampling/createMessage` 支持（server 反过来请求 LLM）。
2. 设计一个 MCP server 的目录结构：包含 tools/、resources/、prompts/ 三个模块。
3. 思考 MCP 的安全边界：如果一个 MCP server 声称自己是「文件管理器」，Agent 如何验证它的能力声明？

> **检查点**：能写一个最小可用的 MCP server（JSON-RPC over stdio），被 client 正确调用。